# Módulo 7 · Clase 6 — Overfitting y Bias-Variance Tradeoff

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | Underfitting vs Overfitting |
| 2 | Bias-Variance Tradeoff |
| 3 | Curvas de aprendizaje |
| 4 | Curvas de validación |
| 5 | Estrategias para corregir cada problema |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, learning_curve, validation_curve
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, accuracy_score

np.random.seed(42)
print("Librerías listas")

---
## 1. Underfitting vs Overfitting

```
UNDERFITTING (sesgo alto):
  El modelo es demasiado simple para capturar los patrones del dataset.
  → Error alto en train Y en test.
  → El modelo "no aprende".
  Ejemplo: regresión lineal en datos cuadráticos.

BUEN AJUSTE:
  El modelo captura los patrones del dataset pero no el ruido.
  → Error bajo en train, error bajo y parecido en test.

OVERFITTING (varianza alta):
  El modelo memoriza el ruido del conjunto de entrenamiento.
  → Error muy bajo en train, error ALTO en test.
  → El modelo "memoriza en lugar de aprender".
  Ejemplo: árbol sin límite de profundidad.

La señal de alerta:
  gap grande entre métrica_train y métrica_test → OVERFITTING
  ambas métricas malas → UNDERFITTING
```

In [ ]:
# Visualizar los 3 casos
np.random.seed(42)
x = np.linspace(0, 10, 60)
y_true = np.sin(x) * 2
y_noisy = y_true + np.random.normal(0, 0.5, len(x))

X_t = x.reshape(-1, 1)
X_tr, X_te, y_tr, y_te = train_test_split(X_t, y_noisy, test_size=0.3, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
x_rango = np.linspace(0, 10, 300).reshape(-1, 1)

configuraciones = [
    ('Underfitting (grado=1)', 1, 'Underfitting'),
    ('Buen ajuste (grado=4)', 4, 'Buen ajuste'),
    ('Overfitting (grado=15)', 15, 'Overfitting'),
]

for ax, (titulo, grado, tipo) in zip(axes, configuraciones):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(grado, include_bias=False)),
        ('reg',  LinearRegression())
    ])
    pipe.fit(X_tr, y_tr)
    rmse_tr = np.sqrt(mean_squared_error(y_tr, pipe.predict(X_tr)))
    rmse_te = np.sqrt(mean_squared_error(y_te, pipe.predict(X_te)))

    ax.scatter(X_tr, y_tr, alpha=0.5, color='steelblue', s=20, label='Train')
    ax.scatter(X_te, y_te, alpha=0.7, color='orange', s=25, marker='x', label='Test')
    ax.plot(x_rango, pipe.predict(x_rango), 'r-', lw=2)
    ax.plot(x_rango, np.sin(x_rango.ravel())*2, 'g--', lw=1, alpha=0.7, label='Función real')
    ax.set_title(f'{titulo}\nRMSE train={rmse_tr:.3f}, test={rmse_te:.3f}')
    ax.legend(fontsize=8)
    ax.set_ylim(-4, 4)

plt.suptitle('Underfitting, Buen Ajuste y Overfitting', fontsize=12)
plt.tight_layout(); plt.show()

---
## 2. Bias-Variance Tradeoff

```
Error total = Bias² + Varianza + Ruido irreducible

BIAS (sesgo):
  Error sistemático por suposiciones erróneas del modelo.
  Modelo simple → bias alto → underfitting

VARIANZA:
  Sensibilidad del modelo a pequeñas variaciones en los datos.
  Modelo complejo → varianza alta → overfitting

TRADEOFF: no se puede reducir ambos a la vez con el mismo modelo.

         Bias²    Varianza    Error
Simple:  ALTO     bajo        alto (underfitting)
Óptimo:  medio    medio       mínimo
Complejo:bajo     ALTO        alto (overfitting)

SOLUCIÓN: ensemble methods (Random Forest, Gradient Boosting)
  → Reducen la varianza manteniendo bias bajo.
```

In [ ]:
# Demostrar bias vs varianza con distintos modelos
np.random.seed(42)

def generar_datos(n=50, seed=None):
    rng = np.random.default_rng(seed)
    x   = rng.uniform(0, 10, n)
    y   = np.sin(x) * 2 + rng.normal(0, 0.5, n)
    return x.reshape(-1,1), y

# Simular 50 experimentos con distintas muestras
n_experimentos = 50
x_test = np.linspace(0, 10, 100).reshape(-1, 1)
y_real = np.sin(x_test.ravel()) * 2

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colores = ['steelblue', 'green', 'coral']
titulos_mod = ['Grado 1 (alto bias)', 'Grado 4 (equilibrado)', 'Grado 12 (alta varianza)']

for ax, grado, titulo, color in zip(axes, [1, 4, 12], titulos_mod, colores):
    predicciones = []
    for i in range(n_experimentos):
        X_i, y_i = generar_datos(n=50, seed=i)
        pipe = Pipeline([
            ('poly', PolynomialFeatures(grado, include_bias=False)),
            ('reg',  Ridge(alpha=0.01))
        ])
        pipe.fit(X_i, y_i)
        predicciones.append(pipe.predict(x_test))

    preds = np.array(predicciones)
    pred_media = preds.mean(axis=0)
    bias2   = np.mean((pred_media - y_real) ** 2)
    varianza = np.mean(preds.var(axis=0))

    for pred in predicciones[:20]:
        ax.plot(x_test, pred, color=color, alpha=0.1, lw=0.8)
    ax.plot(x_test, pred_media, color='black', lw=2, label='Media predicciones')
    ax.plot(x_test, y_real, 'r--', lw=2, label='Función real')
    ax.set_title(f'{titulo}\nBias²={bias2:.3f}, Varianza={varianza:.3f}')
    ax.legend(fontsize=7); ax.set_ylim(-5, 5)

plt.suptitle('Bias-Variance: 50 experimentos por modelo', fontsize=11)
plt.tight_layout(); plt.show()

---
## 3. Curvas de aprendizaje

```
La curva de aprendizaje muestra el error (o score) en función del
tamaño del conjunto de entrenamiento.

Cómo leerla:

UNDERFITTING:                OVERFITTING:               BUEN AJUSTE:
  Train ─── alto               Train ─── bajo             Train ─── bajo
  Test  ─── alto               Test  ─── alto             Test  ─── bajo ≈ Train
  gap pequeño                  gap grande                 gap pequeño

¿Cómo corregir?
  Underfitting → modelo más complejo, más features, menos regularización
  Overfitting  → más datos, menos features, más regularización, ensemble
```

In [ ]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_features=20, n_informative=10,
                           random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
modelos_lc = [
    ('Underfitting\n(árbol depth=1)', DecisionTreeClassifier(max_depth=1, random_state=42)),
    ('Overfitting\n(árbol sin límite)', DecisionTreeClassifier(max_depth=None, random_state=42)),
    ('Equilibrado\n(Random Forest)', RandomForestClassifier(50, max_depth=5, random_state=42, n_jobs=-1)),
]

for ax, (titulo, m) in zip(axes, modelos_lc):
    train_sizes, train_scores, test_scores = learning_curve(
        m, X, y,
        train_sizes=np.linspace(0.1, 1.0, 8),
        cv=5, scoring='accuracy', n_jobs=-1
    )
    train_mean = train_scores.mean(axis=1)
    test_mean  = test_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    test_std   = test_scores.std(axis=1)

    ax.plot(train_sizes, train_mean, 'b-o', ms=5, lw=2, label='Train')
    ax.plot(train_sizes, test_mean,  'r-o', ms=5, lw=2, label='Validación CV')
    ax.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=0.15, color='blue')
    ax.fill_between(train_sizes, test_mean-test_std,  test_mean+test_std,  alpha=0.15, color='red')
    ax.set_ylim(0.5, 1.05)
    ax.set_xlabel('Nº de muestras de entrenamiento')
    ax.set_ylabel('Accuracy')
    ax.set_title(titulo)
    ax.legend(); ax.grid(True, alpha=0.3)
    # Diagnóstico automático
    gap_final = train_mean[-1] - test_mean[-1]
    nivel_train = train_mean[-1]
    if nivel_train < 0.75:
        diag = 'UNDERFITTING'
    elif gap_final > 0.1:
        diag = 'OVERFITTING'
    else:
        diag = 'BUEN AJUSTE'
    ax.text(0.05, 0.05, diag, transform=ax.transAxes, fontsize=11,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.suptitle('Curvas de aprendizaje para diagnóstico de overfitting', fontsize=11)
plt.tight_layout(); plt.show()

---
## 4. Curvas de validación

In [ ]:
# Curva de validación: score vs hiperparámetro
profundidades = [1, 2, 3, 4, 5, 6, 8, 10, 15, 20]

train_scores, test_scores = validation_curve(
    DecisionTreeClassifier(random_state=42),
    X, y,
    param_name='max_depth',
    param_range=profundidades,
    cv=5, scoring='accuracy', n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
test_mean  = test_scores.mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(profundidades, train_mean, 'b-o', ms=5, lw=2, label='Train')
axes[0].plot(profundidades, test_mean,  'r-o', ms=5, lw=2, label='Validación CV')
mejor_depth = profundidades[np.argmax(test_mean)]
axes[0].axvline(mejor_depth, color='green', ls='--', lw=2,
               label=f'Mejor depth={mejor_depth}')
axes[0].set_xlabel('max_depth'); axes[0].set_ylabel('Accuracy')
axes[0].set_title('Curva de validación — max_depth')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Gap train-test
gap = train_mean - test_mean
axes[1].plot(profundidades, gap, 'purple', lw=2, marker='o', ms=5)
axes[1].axhline(0.1, color='red', ls='--', lw=1, label='Umbral overfitting (0.1)')
axes[1].axvline(mejor_depth, color='green', ls='--', lw=1.5)
axes[1].fill_between(profundidades, 0, gap, alpha=0.2, color='purple')
axes[1].set_xlabel('max_depth'); axes[1].set_ylabel('Gap (train - test)')
axes[1].set_title('Gap de generalización')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print(f"Profundidad óptima (mayor score CV): {mejor_depth}")
print(f"  Train: {train_mean[profundidades.index(mejor_depth)]:.4f}")
print(f"  Test CV: {test_mean[profundidades.index(mejor_depth)]:.4f}")

---
## 5. Estrategias de corrección

In [ ]:
print("=" * 60)
print("  DIAGNÓSTICO Y SOLUCIONES")
print("=" * 60)

print("""
UNDERFITTING (bias alto):
  Síntoma: error alto en train Y test, gap pequeño
  Soluciones:
    1. Modelo más complejo (más capas, más árboles, mayor depth)
    2. Más features (ingeniería de features, features polinómicas)
    3. Menos regularización (menor alpha en Ridge/Lasso)
    4. Más tiempo de entrenamiento (más iteraciones)

OVERFITTING (varianza alta):
  Síntoma: error bajo en train, ALTO en test (gap grande)
  Soluciones:
    1. Más datos (la solución más efectiva)
    2. Regularización (Ridge, Lasso, Dropout en redes)
    3. Reducir complejidad (menos depth, menos features)
    4. Ensemble methods (Random Forest, Gradient Boosting)
    5. Early stopping (detener entrenamiento antes)
    6. Data augmentation (crear datos sintéticos)
""")

# Demostración práctica: corregir overfitting
np.random.seed(42)
X_small, y_small = make_classification(n_samples=100, n_features=20,
                                        n_informative=5, random_state=42)
X_large, y_large = make_classification(n_samples=2000, n_features=20,
                                        n_informative=5, random_state=42)

arbol = DecisionTreeClassifier(max_depth=None, random_state=42)

print("Demostración: corregir overfitting")
print(f"{'Estrategia':35} {'Acc_train':>10} {'Acc_test':>10}")

# 1. Baseline (overfitting)
X_tr, X_te, y_tr, y_te = train_test_split(X_small, y_small, test_size=0.3, random_state=42)
arbol.fit(X_tr, y_tr)
print(f"{'Árbol sin límite (pocos datos)':35} "
      f"{accuracy_score(y_tr,arbol.predict(X_tr)):>10.4f} "
      f"{accuracy_score(y_te,arbol.predict(X_te)):>10.4f}")

# 2. Limitar profundidad
arbol_lim = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_tr, y_tr)
print(f"{'Árbol depth=4':35} "
      f"{accuracy_score(y_tr,arbol_lim.predict(X_tr)):>10.4f} "
      f"{accuracy_score(y_te,arbol_lim.predict(X_te)):>10.4f}")

# 3. Más datos
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_large, y_large, test_size=0.3, random_state=42)
arbol.fit(X_tr2, y_tr2)
print(f"{'Árbol sin límite (muchos datos)':35} "
      f"{accuracy_score(y_tr2,arbol.predict(X_tr2)):>10.4f} "
      f"{accuracy_score(y_te2,arbol.predict(X_te2)):>10.4f}")

# 4. Random Forest
rf = RandomForestClassifier(100, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
print(f"{'Random Forest (pocos datos)':35} "
      f"{accuracy_score(y_tr,rf.predict(X_tr)):>10.4f} "
      f"{accuracy_score(y_te,rf.predict(X_te)):>10.4f}")

---
## Práctica

### Ejercicio 1 — Diagnosticar un modelo

**Enunciado:** Dado este modelo y dataset:

```python
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
```

1. Traza la curva de aprendizaje para: DecisionTree depth=1, depth=5, depth=None, RandomForest 100
2. Diagnostica cada modelo (underfitting / overfitting / buen ajuste)
3. ¿Cuál recomendarías para producción?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

data = load_breast_cancer()
X, y = data.data, data.target

modelos = [
    ('DT depth=1',    DecisionTreeClassifier(max_depth=1, random_state=42)),
    ('DT depth=5',    DecisionTreeClassifier(max_depth=5, random_state=42)),
    ('DT sin límite', DecisionTreeClassifier(random_state=42)),
    ('RF 100',        RandomForestClassifier(100, random_state=42, n_jobs=-1)),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, (titulo, m) in zip(axes, modelos):
    sizes, tr_sc, te_sc = learning_curve(
        m, X, y, train_sizes=np.linspace(0.1,1,8), cv=5, scoring='accuracy', n_jobs=-1)
    tr_m, te_m = tr_sc.mean(axis=1), te_sc.mean(axis=1)
    ax.plot(sizes, tr_m, 'b-o', ms=4, label='Train')
    ax.plot(sizes, te_m, 'r-o', ms=4, label='Val CV')
    ax.set_title(titulo); ax.set_ylim(0.7, 1.02)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    # Diagnóstico
    gap = tr_m[-1] - te_m[-1]
    if tr_m[-1] < 0.85: diag = 'UNDERFITTING'
    elif gap > 0.08:    diag = 'OVERFITTING'
    else:               diag = 'BUEN AJUSTE'
    ax.text(0.05, 0.05, diag, transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.suptitle('Diagnóstico: Breast Cancer Dataset', fontsize=11)
plt.tight_layout(); plt.show()
print("Recomendación: Random Forest — buen score, sin overfitting significativo.")
```
</details>

---
### Ejercicio 2 — Curva de validación para Ridge

**Enunciado:** Genera un dataset de regresión con 50 features (30 irrelevantes):

```python
from sklearn.datasets import make_regression
X, y = make_regression(n_samples=200, n_features=50, n_informative=20,
                       noise=20, random_state=42)
```

1. Traza la curva de validación para Ridge con alphas: `[0.001, 0.01, 0.1, 1, 10, 100, 1000]`
2. Identifica el alpha óptimo
3. Compara Ridge óptimo vs LinearRegression: R² en test
4. Muestra el gap train-test para cada alpha

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, validation_curve
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import r2_score

X, y = make_regression(n_samples=200, n_features=50, n_informative=20, noise=20, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
tr_sc, te_sc = validation_curve(
    Ridge(), X_tr, y_tr,
    param_name='alpha', param_range=alphas,
    cv=5, scoring='r2', n_jobs=-1
)

tr_m, te_m = tr_sc.mean(axis=1), te_sc.mean(axis=1)
mejor_alpha = alphas[np.argmax(te_m)]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].semilogx(alphas, tr_m, 'b-o', ms=5, label='Train')
axes[0].semilogx(alphas, te_m, 'r-o', ms=5, label='Val CV')
axes[0].axvline(mejor_alpha, color='green', ls='--', label=f'α óptimo={mejor_alpha}')
axes[0].set_xlabel('Alpha'); axes[0].set_ylabel('R²')
axes[0].set_title('Curva de validación Ridge'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

gap = tr_m - te_m
axes[1].semilogx(alphas, gap, 'purple', lw=2, marker='o', ms=5)
axes[1].axvline(mejor_alpha, color='green', ls='--')
axes[1].set_xlabel('Alpha'); axes[1].set_ylabel('Gap train-test')
axes[1].set_title('Gap de generalización'); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Comparativa en test
for nombre, m in [('LinearRegression', LinearRegression()),
                  (f'Ridge α={mejor_alpha}', Ridge(alpha=mejor_alpha))]:
    m.fit(X_tr, y_tr)
    r2 = r2_score(y_te, m.predict(X_te))
    r2_tr = r2_score(y_tr, m.predict(X_tr))
    print(f"{nombre:25}: R²_train={r2_tr:.4f}, R²_test={r2:.4f}")
print(f"\nRidge óptimo reduce el gap gracias a la regularización.")
```
</details>

---
### Ejercicio 3 — ¿Más datos o modelo más simple?

**Enunciado:** Una empresa tiene 200 muestras y su árbol (depth=10) tiene overfitting. Antes de recoger más datos (caro), quiere saber si la solución más sencilla (limitar el árbol) funciona.

```python
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=2000, n_features=20, n_informative=8, random_state=42)
X_200, _, y_200, _ = train_test_split(X, y, train_size=200, random_state=42)
X_te, y_te = X[1800:], y[1800:]  # test fijo
```

1. Traza curvas de aprendizaje para árbol depth=10 y RF 100
2. ¿A partir de cuántas muestras el árbol depth=10 deja de tener overfitting?
3. ¿Qué recomiendas: más datos o Random Forest con 200 muestras?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=2000, n_features=20, n_informative=8, random_state=42)

# Curvas de aprendizaje con todos los datos disponibles
modelos = [
    ('Árbol depth=10', DecisionTreeClassifier(max_depth=10, random_state=42)),
    ('RF 100', RandomForestClassifier(100, random_state=42, n_jobs=-1)),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, (titulo, m) in zip(axes, modelos):
    sizes, tr_sc, te_sc = learning_curve(
        m, X, y, train_sizes=np.linspace(0.05, 0.9, 10),
        cv=5, scoring='accuracy', n_jobs=-1)
    tr_m, te_m = tr_sc.mean(axis=1), te_sc.mean(axis=1)
    ax.plot(sizes, tr_m, 'b-o', ms=4, label='Train')
    ax.plot(sizes, te_m, 'r-o', ms=4, label='Val CV')
    # Marcar dónde el gap cae bajo 0.05
    gaps = tr_m - te_m
    bajo_umbral = np.where(gaps < 0.05)[0]
    if len(bajo_umbral) > 0:
        n_umbral = sizes[bajo_umbral[0]]
        ax.axvline(n_umbral, color='green', ls='--', label=f'Gap<5% desde n={n_umbral:.0f}')
    ax.set_title(titulo); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.set_xlabel('Nº muestras'); ax.set_ylabel('Accuracy')

plt.tight_layout(); plt.show()

# Comparativa con 200 muestras
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
X_200 = X_tr[:200]; y_200 = y_tr[:200]

for nombre, m in modelos:
    m.fit(X_200, y_200)
    acc = accuracy_score(y_te, m.predict(X_te))
    print(f"{nombre}: test accuracy con 200 muestras = {acc:.4f}")

print("\nRecomendación: RF con 200 muestras suele ser mejor que árbol.")
print("El bagging compensa la escasez de datos.")
```
</details>

---
### Ejercicio 4 — Receta de diagnóstico

**Enunciado:** Dado este modelo en producción que reporta problemas:

```python
from sklearn.datasets import make_regression
X, y = make_regression(n_samples=150, n_features=30, n_informative=5, noise=30, random_state=7)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
from sklearn.linear_model import LinearRegression
m = LinearRegression().fit(X_tr, y_tr)
```

1. Calcula R² en train y test → ¿cuál es el diagnóstico?
2. Traza la curva de aprendizaje
3. Propón y prueba 2 soluciones distintas
4. ¿Cuál mejora más el R² en test?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

X, y = make_regression(n_samples=150, n_features=30, n_informative=5, noise=30, random_state=7)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)

m = LinearRegression().fit(X_tr, y_tr)

# 1. Diagnóstico
r2_tr = r2_score(y_tr, m.predict(X_tr))
r2_te = r2_score(y_te, m.predict(X_te))
print(f"R² train: {r2_tr:.4f}")
print(f"R² test:  {r2_te:.4f}")
print(f"Gap:      {r2_tr - r2_te:.4f}")
print("→ OVERFITTING: R² muy bajo en test vs train")
print("  Causa: n=150, p=30 → muchas features relativas a las muestras")

# 2. Curva de aprendizaje
sizes, tr_sc, te_sc = learning_curve(m, X, y, train_sizes=np.linspace(0.1,0.9,8),
                                      cv=5, scoring='r2', n_jobs=-1)
plt.figure(figsize=(8,4))
plt.plot(sizes, tr_sc.mean(1), 'b-o', label='Train')
plt.plot(sizes, te_sc.mean(1), 'r-o', label='Val CV')
plt.xlabel('Muestras'); plt.ylabel('R²'); plt.title('Curva de aprendizaje')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# 3. Soluciones
for nombre, mod in [
    ('LinearRegression', LinearRegression()),
    ('Ridge α=10',       Ridge(alpha=10)),
    ('Lasso α=1',        Lasso(alpha=1)),
    ('RandomForest',     RandomForestRegressor(100, random_state=42, n_jobs=-1)),
]:
    mod.fit(X_tr, y_tr)
    r2 = r2_score(y_te, mod.predict(X_te))
    r2_tr_i = r2_score(y_tr, mod.predict(X_tr))
    print(f"{nombre:20}: R²_train={r2_tr_i:.4f}, R²_test={r2:.4f}")
```
</details>

---
### Ejercicio 5 (reto) — Función diagnóstico automático

**Enunciado:** Implementa `diagnosticar_modelo(modelo, X, y, cv=5)` que:

1. Calcula y plotea la curva de aprendizaje
2. Diagnostica automáticamente: underfitting / overfitting / buen ajuste
3. Propone las dos soluciones más adecuadas según el diagnóstico
4. Devuelve un dict con `{'diagnostico': ..., 'score_train': ..., 'score_val': ..., 'gap': ..., 'soluciones': [...]}`

Pruébalo con 3 modelos distintos.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve

SOLUCIONES = {
    'underfitting': [
        'Modelo más complejo (más profundidad, más árboles)',
        'Más features o features polinómicas',
        'Menos regularización (menor alpha)',
    ],
    'overfitting': [
        'Recoger más datos de entrenamiento',
        'Aumentar regularización (mayor alpha)',
        'Reducir complejidad del modelo',
        'Usar ensemble (Random Forest)',
    ],
    'buen ajuste': [
        'El modelo generaliza bien',
        'Considera ajustar el umbral de clasificación si es relevante',
    ],
}

def diagnosticar_modelo(modelo, X, y, cv=5, scoring='accuracy', ax=None):
    sizes, tr_sc, te_sc = learning_curve(
        modelo, X, y,
        train_sizes=np.linspace(0.1, 1.0, 8),
        cv=cv, scoring=scoring, n_jobs=-1
    )
    tr_m, te_m = tr_sc.mean(axis=1), te_sc.mean(axis=1)
    score_train = tr_m[-1]
    score_val   = te_m[-1]
    gap         = score_train - score_val

    # Diagnóstico
    umbral_malo = 0.75 if scoring == 'accuracy' else 0.5
    if score_train < umbral_malo:   diag = 'underfitting'
    elif gap > 0.08:                diag = 'overfitting'
    else:                           diag = 'buen ajuste'

    # Plot
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4))
    ax.plot(sizes, tr_m, 'b-o', ms=5, lw=2, label='Train')
    ax.plot(sizes, te_m, 'r-o', ms=5, lw=2, label='Val CV')
    ax.fill_between(sizes, tr_sc.min(1), tr_sc.max(1), alpha=0.1, color='blue')
    ax.fill_between(sizes, te_sc.min(1), te_sc.max(1), alpha=0.1, color='red')
    ax.set_title(f'{type(modelo).__name__}\n{diag.upper()}')
    ax.set_xlabel('Muestras'); ax.set_ylabel(scoring)
    ax.legend(); ax.grid(True, alpha=0.3)
    ax.text(0.02, 0.05, diag.upper(), transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat'))

    return {
        'diagnostico': diag,
        'score_train': round(score_train, 4),
        'score_val':   round(score_val, 4),
        'gap':         round(gap, 4),
        'soluciones':  SOLUCIONES[diag][:2],
    }

# Prueba
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

X, y = make_classification(n_samples=500, n_features=20, n_informative=8, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, m in zip(axes, [
    DecisionTreeClassifier(max_depth=1, random_state=42),
    DecisionTreeClassifier(max_depth=None, random_state=42),
    RandomForestClassifier(100, random_state=42, n_jobs=-1),
]):
    resultado = diagnosticar_modelo(m, X, y, ax=ax)
    print(f"\n{type(m).__name__}:")
    for k, v in resultado.items():
        print(f"  {k}: {v}")

plt.tight_layout(); plt.show()
```
</details>

---
## Resumen

```python
from sklearn.model_selection import learning_curve, validation_curve
import numpy as np

# Curva de aprendizaje (diagnóstico por tamaño de datos)
sizes, tr_scores, te_scores = learning_curve(
    modelo, X, y,
    train_sizes=np.linspace(0.1, 1.0, 8),
    cv=5, scoring='accuracy'
)

# Curva de validación (diagnóstico por hiperparámetro)
tr_scores, te_scores = validation_curve(
    modelo, X, y,
    param_name='max_depth', param_range=[1,2,3,5,10],
    cv=5, scoring='accuracy'
)

# Diagnóstico rápido:
# score_train alto, score_val bajo  → OVERFITTING
# ambos bajos                        → UNDERFITTING
# ambos altos y parecidos            → BUEN AJUSTE
```

## ¿Qué viene en la próxima clase?
En la **Clase 7** aprenderemos **Cross-Validation** y **GridSearchCV**: las herramientas estándar para evaluar modelos y encontrar los mejores hiperparámetros de forma fiable.